In [ ]:
%pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# The exact path where the unzipped files ended up
# (Notice the repeated kaggle/working due to absolute path zipping)
SAVED_MODEL_PATH = "/kaggle/input/models/komvopoulos/modeling/other/q3.5-2b/1/kaggle/working/Qwen3.5-2B"

print(f"Loading model from {SAVED_MODEL_PATH}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SAVED_MODEL_PATH, # <--- POINT TO YOUR LOCAL FOLDER
    max_seq_length=512,          # Keep this matching your generation limit
    dtype=torch.float16,         # Use 16-bit for inference
    load_in_4bit=True,           # Keep 4-bit to save RAM
)

FastLanguageModel.for_inference(model)
print("Model loaded and ready for inference!")

In [ ]:
import time
import re
import pandas as pd
import torch
import xml.etree.ElementTree as ET

# 1. SVG UTILS
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

# 2. BATCHED GENERATION SETUP
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# IMPORTANT: Make sure SYSTEM_PROMPT is defined somewhere above!
SYSTEM_PROMPT = "You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element."

def generate_svg_batch(prompts, max_new_tokens=512):
    chat_texts = [
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
        for p in prompts
    ]
    
    inputs = tokenizer(text=chat_texts, return_tensors="pt", padding=True).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            # temperature=0.7,
            # top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )
    
    input_length = inputs.input_ids.shape[1]
    generated_tokens = output_ids[:, input_length:]
    decoded_batch = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    
    svgs = []
    for i, decoded in enumerate(decoded_batch):
        svg = extract_svg(decoded)
        if not is_valid_svg(svg):
            svg = fallback_svg(prompts[i])
        svgs.append(svg)
        
    return svgs


# 3. SUBMISSION GENERATION LOOP WITH TIMER
TEST_PROMPTS_PATH = "/kaggle/input/datasets/komvopoulos/testing/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

print("\nStarting batched generation on test set...")
test_df = pd.read_csv(TEST_PROMPTS_PATH)

BATCH_SIZE = 16
rows = []
invalid_count = 0
total_rows = len(test_df)
t0 = time.time()

for i in range(0, total_rows, BATCH_SIZE):
    batch_t0 = time.time() 
    
    batch_df = test_df.iloc[i : i + BATCH_SIZE]
    prompts = batch_df["prompt"].tolist()
    ids = batch_df["id"].tolist()
    
    batch_svgs = generate_svg_batch(prompts)
    
    for j in range(len(batch_svgs)):
        svg = batch_svgs[j]
        if 'circle cx="128"' in svg and 'fill="black"' in svg:
            invalid_count += 1
        rows.append({"id": ids[j], "svg": svg})
    
    torch.cuda.empty_cache()
    
    batch_time = time.time() - batch_t0
    processed = min(i + BATCH_SIZE, total_rows)
    elapsed_time = time.time() - t0
    avg_time_per_batch = elapsed_time / (processed / BATCH_SIZE)
    batches_left = (total_rows - processed) / BATCH_SIZE
    eta_mins = (batches_left * avg_time_per_batch) / 60
    
    print(f"Processed {processed}/{total_rows} | Batch Time: {batch_time:.2f}s | ETA: {eta_mins:.1f} mins")

# 4. SAVE AND FINISH
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\nSaved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Total Runtime (minutes): {elapsed_min:.2f}")

In [3]:
import pandas as pd
import xml.etree.ElementTree as ET
import re

# 1. Define the exact competition rules
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", 
    "polygon", "defs", "use", "symbol", "clipPath", "mask", "linearGradient", 
    "radialGradient", "stop", "text", "tspan", "title", "desc", "style", 
    "pattern", "marker", "filter"
}

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def heal_svg(svg_str):
    if not svg_str or not isinstance(svg_str, str):
        return fallback_svg(), "Empty"
    
    svg_str = svg_str.strip()
    
    # --- HEAL 1: Fix Hallucinated Tags ---
    # Replace <rectangle> with <rect>
    svg_str = svg_str.replace("<rectangle", "<rect").replace("</rectangle>", "</rect>")
    
    # --- HEAL 2: Force the correct Canvas dimensions ---
    # Strip out whatever viewBox, width, or height the model hallucinated
    svg_str = re.sub(r'(?i)\b(viewBox|width|height)=["\'][^"\']*["\']', '', svg_str)
    
    # Inject the exact competition required dimensions into the opening <svg> tag
    svg_str = svg_str.replace("<svg", '<svg width="256" height="256" viewBox="0 0 256 256" ', 1)
    
    # Now check if it passes the strict rules
    try:
        root = ET.fromstring(svg_str)
    except ET.ParseError:
        return fallback_svg(), "Invalid XML structure"
    
    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        if tag not in ALLOWED_TAGS:
            return fallback_svg(), f"Disallowed tag: <{tag}>"
            
        if tag == "path":
            path_count += 1
            
    if path_count > 256:
        return fallback_svg(), f"Too many paths: {path_count}"
        
    if len(svg_str) > 8000:
        return fallback_svg(), "Too long (>8000 chars)"

    return svg_str, "Healed"

# 2. Load the original generated submission
SUBMISSION_PATH = "/kaggle/input/datasets/komvopoulos/submission/submission (8).csv"
print(f"Loading {SUBMISSION_PATH}...")
df = pd.read_csv(SUBMISSION_PATH)

# 3. Heal the submission
healed_count = 0
destroyed_count = 0

for idx, row in df.iterrows():
    # Only try to heal it if it isn't already our fallback circle
    if 'circle cx="128"' not in row['svg'] or 'fill="black"' not in row['svg']:
        healed_svg, status = heal_svg(row['svg'])
        
        if status == "Healed":
            # We successfully saved the model's drawing!
            df.at[idx, 'svg'] = healed_svg
            healed_count += 1
        else:
            # It was completely unfixable (e.g., <move> tag), apply fallback
            df.at[idx, 'svg'] = fallback_svg()
            destroyed_count += 1

# 4. Save the cleaned version
CLEANED_SUBMISSION_PATH = "/kaggle/working/submission_healed.csv"
df.to_csv(CLEANED_SUBMISSION_PATH, index=False)

print(f"\nDone!")
print(f"Successfully rescued & healed {healed_count} SVGs!")
print(f"Unfixable SVGs replaced with fallback: {destroyed_count}")
print(f"Healed submission saved to: {CLEANED_SUBMISSION_PATH}")

Loading /kaggle/input/datasets/komvopoulos/submission/submission (8).csv...

Done!
Successfully rescued & healed 369 SVGs!
Unfixable SVGs replaced with fallback: 59
Healed submission saved to: /kaggle/working/submission_healed.csv
